In [ ]:
!pip install pennylane


In [21]:
import pennylane as qml
from pennylane import numpy as np
import itertools
import matplotlib.pyplot as plt

np.random.seed(42)

In [22]:
N = 12
J = 1.0
h= 0.5
layers = 4

dev = qml.device("default.qubit", wires=N)

In [25]:
dev = qml.device("default.qubit", wires=N)
def quantum_ising_hamiltonian(N, J, h):

    coeffs = []
    obs = []

    # ZZ
    for i in range(N - 1):
        coeffs.append(-J)
        obs.append(qml.PauliZ(i) @ qml.PauliZ(i+1))

    # X (campo transversal)
    for i in range(N):
        coeffs.append(-h)
        obs.append(qml.PauliX(i))

    return qml.Hamiltonian(coeffs, obs)

H = quantum_ising_hamiltonian(N, J, h)

In [23]:
layers = 2
num_params = layers * N

@qml.qnode(dev)
def vqe_circuit(params):

    idx = 0

    for l in range(layers):

        # rotaciones locales
        for i in range(N):
            qml.RY(params[idx], wires=i)
            idx += 1

        # entrelazamiento
        for i in range(N - 1):
            qml.CNOT(wires=[i, i+1])

    return qml.expval(H)

In [26]:
def cost_vqe(params):
    return vqe_circuit(params)

params = 0.1 * np.random.randn(num_params, requires_grad=True)

opt = qml.AdamOptimizer(0.05)

for i in range(200):

    params = opt.step(cost_vqe, params)

    if i % 20 == 0:
        print(i, cost_vqe(params))

0 -10.824606903072665
20 -11.332187412759987
40 -11.377974036066183
60 -11.38865439869282
80 -11.397953316060603
100 -11.415241712706978
120 -11.437325311691529
140 -11.472962477946703
160 -11.53579914424074
180 -11.643992498906345


In [27]:
layers_mps = 3
num_params_mps = layers_mps * N

@qml.qnode(dev)
def mps_circuit(params):

    idx = 0

    for l in range(layers_mps):

        # rotaciones locales
        for i in range(N):
            qml.RY(params[idx], wires=i)
            idx += 1

        # forward chain
        for i in range(N - 1):
            qml.CNOT(wires=[i, i+1])

        # backward chain (clave MPS)
        for i in reversed(range(N - 1)):
            qml.CNOT(wires=[i+1, i])

    return qml.expval(H)

In [28]:
def cost_mps(params):
    return mps_circuit(params)

params_mps = 0.1 * np.random.randn(num_params_mps, requires_grad=True)

opt = qml.AdamOptimizer(0.05)

for i in range(200):

    params_mps = opt.step(cost_mps, params_mps)

    if i % 20 == 0:
        print(i, cost_mps(params_mps))

0 -10.824871646363553
20 -11.22933553525545
40 -11.258030732667182
60 -11.267742933071103
80 -11.295090287824596
100 -11.351368004007396
120 -11.401549469232252
140 -11.43660680609629
160 -11.477989866446567
180 -11.57857340806677


In [29]:
E_vqe = cost_vqe(params)
E_mps = cost_mps(params_mps)

print("VQE:", E_vqe)
print("MPS-like:", E_mps)

VQE: -11.726054515014864
MPS-like: -11.681421250523426


In [30]:
import numpy as np
import scipy.linalg as la

def exact_quantum_energy(H):
    H_matrix = qml.matrix(H)
    eigvals = la.eigvalsh(H_matrix)
    return np.min(eigvals)

E_exact_quantum = exact_quantum_energy(H)
print(E_exact_quantum)

-11.892044872938808
